# 03 — Multiple Exceptions

## Exception Handling

This notebook builds directly on `02_try_and_except.ipynb`.

The main focus is:

> **How to handle different types of exceptions separately and correctly.**

### Learning goals

By the end of this notebook, you should understand:

- why multiple exception handlers are needed
- how multiple `except` blocks work
- how Python chooses an `except` block
- why only one matching handler executes
- how to handle different exceptions differently
- how to combine exception types in one `except`
- how to capture exception objects with `as e`
- how a general `Exception` handler can act as a fallback
- why specific exceptions should generally come before broader exceptions
- how exception hierarchy affects handler ordering
- how multiple exception handling can be used in functions and practical programs

> **Boundary:** This notebook focuses on multiple exception handling. `else`, `finally`, `raise`, and custom exceptions are covered later.

## 1. Introduction

The basic pattern from the previous notebook is:

```python
try:
    # risky code
except SomeException:
    # handle exception
```

A single block of code can potentially produce **different types of exceptions**.

For example:

```python
number = int(input("Enter a number: "))
result = 100 / number
```

This code can produce:

- `ValueError` → the input cannot be converted to an integer
- `ZeroDivisionError` → the user enters `0`

Therefore, we may need multiple exception handlers.

## 2. Why Do We Need Multiple Exception Handlers?

Different exceptions can require different responses.

```python
try:
    number = int(input("Enter a number: "))
    result = 100 / number
except ValueError:
    print("Please enter a valid integer")
except ZeroDivisionError:
    print("Number cannot be zero")
```

The flow is:

```text
Input
  │
  ├── invalid integer → ValueError
  │
  ├── 0 → ZeroDivisionError
  │
  └── valid non-zero integer → successful execution
```

Each exception gets its own appropriate response.

In [ ]:
try:
    number = int(input("Enter a number: "))
    result = 100 / number
    print(result)
except ValueError:
    print("Please enter a valid integer")
except ZeroDivisionError:
    print("Number cannot be zero")

## 3. Multiple `except` Blocks

The basic syntax is:

```python
try:
    # risky code
except ExceptionType1:
    # handle type 1
except ExceptionType2:
    # handle type 2
except ExceptionType3:
    # handle type 3
```

Example:

In [ ]:
try:
    value = int(input("Enter a number: "))
    print(100 / value)
except ValueError:
    print("Invalid number")
except ZeroDivisionError:
    print("Cannot divide by zero")

Each `except` block is associated with a particular exception type.

This allows the program to respond differently depending on what went wrong.

## 4. How Python Chooses an `except` Block

Python checks the `except` clauses **in order**.

When an exception occurs:

1. Python checks the first handler.
2. If it does not match, Python checks the next handler.
3. When a matching handler is found, that handler executes.
4. The remaining `except` blocks are skipped.

```text
Exception occurs
      ↓
First except?
   │       │
  Yes      No
   ↓       ↓
Handle   Next except
            ↓
          Match?
```

The order of handlers therefore matters.

In [ ]:
try:
    number = int("abc")
except ValueError:
    print("ValueError handler")
except TypeError:
    print("TypeError handler")

`int("abc")` raises `ValueError`, so the first handler matches and executes.

The `TypeError` handler is not executed.

## 5. Only One Matching `except` Executes

Consider:

In [ ]:
try:
    number = int("abc")
except ValueError:
    print("Value error")
except Exception:
    print("General exception")

Output:

```text
Value error
```

The second handler does not also execute.

Python selects the first matching handler and then continues after the complete `try-except` structure.

## 6. Handling Different Exceptions Differently

Here is a practical example involving two inputs:

In [ ]:
try:
    a = int(input("Enter first number: "))
    b = int(input("Enter second number: "))

    result = a / b

    print(result)

except ValueError:
    print("Please enter integers only")

except ZeroDivisionError:
    print("The second number cannot be zero")

This is one of the most common patterns for multiple exception handling.

- Invalid numeric input produces `ValueError`.
- A zero divisor produces `ZeroDivisionError`.
- Valid, non-zero integers allow the calculation to complete.

## 7. Multiple Exceptions in One `except`

Sometimes different exceptions require the **same response**.

Python allows multiple exception types to be specified in a tuple:

```python
except (ValueError, TypeError):
    print("Invalid input")
```

The tuple:

```python
(ValueError, TypeError)
```

means that either exception can be handled by the same block.

In [ ]:
try:
    value = int("abc")
except (ValueError, TypeError):
    print("Invalid value")

The same handler is used if the exception is either `ValueError` or `TypeError`.

This is useful when the response should be identical.

## 8. Example of a Tuple of Exceptions

A tuple of exception classes is especially useful when different operations can fail in different ways but the program has the same response for both.

In [ ]:
def process_value(value):
    try:
        return int(value)
    except (ValueError, TypeError):
        return "Invalid value"


print(process_value("25"))
print(process_value("abc"))

The function uses one handler for both `ValueError` and `TypeError`.

The important idea is not the specific function, but the syntax:

```python
except (ValueError, TypeError):
```

## 9. Multiple `except` vs Tuple of Exceptions

### Separate handling

```python
try:
    ...
except ValueError:
    print("Value error")
except TypeError:
    print("Type error")
```

Use this when the responses need to be different.

### Combined handling

```python
try:
    ...
except (ValueError, TypeError):
    print("Invalid input")
```

Use this when the handling is the same.

The choice depends on whether the exceptions need different treatment.

## 10. Capturing the Exception Object

The exception object can still be captured with `as e`.

For example:

In [ ]:
try:
    number = int("abc")
except ValueError as e:
    print("Error:", e)

With multiple handlers, each handler can access its own exception object.

In [ ]:
try:
    number = int(input("Enter number: "))
    result = 100 / number

except ValueError as e:
    print("ValueError:", e)

except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

The `e` variable refers to the exception object caught by that particular handler.

This can be useful when the exception message provides additional information about what happened.

## 11. Handling a General Exception

A general fallback can be written as:

```python
except Exception as e:
    print("An unexpected error occurred:", e)
```

Example:

In [ ]:
try:
    number = int(input("Enter a number: "))
    result = 100 / number

except ValueError:
    print("Invalid value")

except ZeroDivisionError:
    print("Cannot divide by zero")

except Exception as e:
    print("Unexpected error:", e)

The `Exception` handler can act as a general fallback for many ordinary exceptions.

However, it should not replace thoughtful specific handling. If you know what exception you expect, handling that specific exception is clearer.

## 12. Specific Exceptions Before General Exceptions

This is **very important**.

### Correct ordering

```python
try:
    ...
except ValueError:
    print("Value error")
except Exception:
    print("Other exception")
```

### Incorrect ordering

```python
try:
    ...
except Exception:
    print("General error")
except ValueError:
    print("Value error")
```

Why?

`ValueError` is a subclass of `Exception`.

Therefore, the broad `Exception` handler can catch a `ValueError` first, preventing the later `ValueError` handler from being reached for that exception.

Specific handlers should generally come before broader handlers.

In [ ]:
try:
    number = int("abc")
except ValueError:
    print("Specific ValueError handler")
except Exception:
    print("General handler")

The `ValueError` handler runs because it appears before the broader `Exception` handler.

## 13. Exception Hierarchy and Handler Ordering

Exception classes form a hierarchy.

A simplified view is:

```text
Exception
   │
   ├── ValueError
   │
   ├── TypeError
   │
   ├── LookupError
   │      ├── IndexError
   │      └── KeyError
   │
   └── ...
```

Therefore:

```python
except ValueError:
```

should generally come before:

```python
except Exception:
```

when both are present.

Similarly:

```python
except IndexError:
```

should generally come before a broader:

```python
except LookupError:
```

if both are present and different handling is intended.

## 14. Example — `IndexError` and `LookupError`

`IndexError` is part of the `LookupError` branch of the exception hierarchy.

Example:

In [ ]:
numbers = [10, 20, 30]

try:
    print(numbers[5])

except IndexError:
    print("Invalid list index")

except LookupError:
    print("Lookup failed")

The `IndexError` handler runs because `IndexError` is the more specific match.

The broader `LookupError` handler is not also executed.

## 15. Example — `KeyError` and `LookupError`

`KeyError` is also related to `LookupError`.

In [ ]:
student = {
    "name": "Komal"
}

try:
    print(student["marks"])

except KeyError:
    print("Key not found")

except LookupError:
    print("Lookup failed")

Again, the specific `KeyError` handler comes first.

This demonstrates why understanding exception hierarchy helps when ordering handlers.

## 16. Nested Exceptions

Exception handling can be nested.

Example:

In [ ]:
try:
    try:
        number = int(input("Enter number: "))
        print(100 / number)

    except ValueError:
        print("Invalid input")

except ZeroDivisionError:
    print("Cannot divide by zero")

The inner `try-except` handles `ValueError`.

If a `ZeroDivisionError` occurs inside the inner `try` and is not handled by the inner `except`, the exception can propagate outward to the outer handler.

Nested exception handling should be used only when the program structure actually requires it. Unnecessary nesting can make code harder to understand.

## 17. Multiple Exceptions in Functions

Exception handling can be used inside functions.

Example:

In [ ]:
def divide_numbers(a, b):
    try:
        return a / b

    except TypeError:
        return "Both values must be numbers"

    except ZeroDivisionError:
        return "Cannot divide by zero"


print(divide_numbers(10, 2))
print(divide_numbers(10, 0))
print(divide_numbers(10, "2"))

The function handles two different exception types:

- `ZeroDivisionError` → division by zero
- `TypeError` → incompatible operand types

This is a practical connection between exception handling and functions.

## 18. Multiple Exceptions with User Input

A complete user-input example:

In [ ]:
try:
    num1 = int(input("Enter first number: "))
    num2 = int(input("Enter second number: "))

    print("Result:", num1 / num2)

except ValueError:
    print("Please enter valid integers.")

except ZeroDivisionError:
    print("Second number cannot be zero.")

There are three important execution cases.

### Valid input

```text
10
2
```

The division succeeds.

### Invalid input

```text
abc
2
```

The integer conversion raises `ValueError`.

### Zero

```text
10
0
```

The division raises `ZeroDivisionError`.

Each problem receives a different response.

## 19. `except Exception as e` as a Fallback

A common pattern is:

```python
try:
    # operation
except SpecificException:
    # expected problem
except AnotherSpecificException:
    # another expected problem
except Exception as e:
    # broad fallback
```

For example:

In [ ]:
try:
    number = int(input("Enter a number: "))
    result = 100 / number

except ValueError:
    print("Invalid integer")

except ZeroDivisionError:
    print("Cannot divide by zero")

except Exception as e:
    print("Unexpected error:", e)

The conceptual flow is:

```text
Specific handlers
       ↓
Expected problems
       ↓
General handler
       ↓
Other ordinary exceptions
```

The general handler should not replace specific handling. It is a fallback when broader handling is genuinely appropriate.

## 20. Common Mistakes

### Mistake 1 — Putting `Exception` first

```python
except Exception:
    ...
except ValueError:
    ...
```

This prevents the specific `ValueError` handler from being reached when a `ValueError` is raised.

Put the more specific handler first.

---

### Mistake 2 — Combining exceptions unnecessarily

Do not combine:

```python
except (ValueError, ZeroDivisionError):
    print("Something went wrong")
```

if the two exceptions actually require different responses.

Use separate handlers when different handling is needed.

---

### Mistake 3 — Separating exceptions unnecessarily

If two exceptions require exactly the same response:

```python
except ValueError:
    print("Invalid input")

except TypeError:
    print("Invalid input")
```

they can potentially be combined:

```python
except (ValueError, TypeError):
    print("Invalid input")
```

---

### Mistake 4 — Using too many exception handlers

Only handle exceptions that you can meaningfully respond to.

---

### Mistake 5 — Assuming all exceptions inherit directly from `Exception`

Python has an exception hierarchy.

Some important classes, such as `SystemExit` and `KeyboardInterrupt`, are under `BaseException` rather than `Exception`.

This notebook only needs the basic awareness that `Exception` is not the root of every exception-related class in Python.

## 21. Practical Example — Simple Calculator

The following example combines multiple exception types in a small calculator.

In [ ]:
try:
    a = float(input("Enter first number: "))
    b = float(input("Enter second number: "))
    operation = input("Enter operation (+, -, *, /): ")

    if operation == "+":
        result = a + b
    elif operation == "-":
        result = a - b
    elif operation == "*":
        result = a * b
    elif operation == "/":
        result = a / b
    else:
        print("Invalid operation")
        result = None

    if result is not None:
        print("Result:", result)

except ValueError:
    print("Please enter valid numbers.")

except ZeroDivisionError:
    print("Cannot divide by zero.")

This example handles:

- invalid numeric input with `ValueError`
- division by zero with `ZeroDivisionError`
- invalid operations through normal conditional logic

Not every invalid situation is an exception. For example, an unsupported operation string is handled here with an ordinary `if` condition.

## 22. Summary

By the end of this notebook, you should understand:

- why multiple exception handlers are needed
- multiple `except` blocks
- the order of `except` blocks
- why only one matching handler executes
- handling different exceptions differently
- combining exception types in one `except`
- `except (ValueError, TypeError)`
- exception objects with `as e`
- the general `Exception` class
- specific-before-general ordering
- basic exception hierarchy and handler matching
- multiple exceptions in functions
- practical user-input handling

### Next notebook

**`04_else_and_finally.ipynb`**

The next notebook will cover what happens when the `try` block succeeds and how `finally` is used for code that must run regardless of whether an exception occurred.